# Run the HIF / CICIDS2017 comparison on Google Colab

This notebook clones the repository, installs the pinned dependencies,
downloads the dataset and runs the full comparison. It is meant to run on
Colab so you do not have to run the heavy pipeline on your own machine.

## Runtime: use CPU, not GPU

The entire pipeline (HIF, Random Forest, MLP, SVM, LOF) runs on CPU through
scikit-learn. There is NO GPU acceleration, so selecting a GPU runtime (T4,
L4, A100) gives zero speedup and only burns compute units. Pick:

    Runtime -> Change runtime type -> CPU, and enable High-RAM.

High-RAM matters because the dataset has millions of rows. More vCPUs help:
Random Forest, the HIF forest, LOF and the SVM calibration folds use all
cores. The MLP is the main step that does not parallelize within a fit; to
use spare cores for it during tuning, add `--parallel_trials 4` to the final
run (this makes the MLP's Optuna search non-deterministic, so it is off by
default).

## 1. Clone the repository
Edit REPO_URL if you forked it or pushed it elsewhere.

In [ ]:
REPO_URL = "https://github.com/Dicotomico23/hif-cicids2017.git"
import os
os.chdir('/content')  # always operate from a fixed base to avoid nested clones
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo):
    !git clone $REPO_URL
else:
    # Already cloned: pull the latest code instead of cloning into a subfolder.
    !cd {repo} && git pull --ff-only
%cd /content/{repo}
!git log --oneline -1

## 2. Install dependencies

In [ ]:
# Colab already ships numpy/pandas/scipy/scikit-learn/matplotlib. Install only
# the missing pieces and the package itself, without forcing version changes.
!pip install -q -U kagglehub imbalanced-learn optuna
!pip install -q -e . --no-deps
# Note: this uses Colab's library versions. For the exact pinned environment
# used in the paper, install requirements.txt locally instead.


## 3. Get the dataset

The next cell runs `data/download.py`, which downloads the cleaned dataset
from this GitHub Release asset and verifies its SHA256 checksum:

https://github.com/Dicotomico23/hif-cicids2017/releases/download/dataset-v1/cicids2017_cleaned.zip

No Kaggle account is needed. Kaggle stays as an optional emergency fallback
(commented at the bottom of the cell).

In [ ]:
!python data/download.py
import os
assert os.path.isfile('data/cicids2017_cleaned.csv'), (
    'Cleaned dataset is missing. Re-run this cell before the comparison; '
    'otherwise the pipeline falls back to the RAW Kaggle data (78 features '
    'instead of 52) and the numbers will not match the paper.')
print('Cleaned dataset ready (%.0f MB)'
      % (os.path.getsize('data/cicids2017_cleaned.csv') / 1e6))

## 4. Run the comparison

`reproduce/run_comparison.py` runs the whole pipeline (load, clean, split,
feature selection, scaling, Borderline-SMOTE, HIF ensemble, supervised
baselines, LOF) and writes the table and figures to `--output`.

### All command-line flags

| Flag | Default | What it does |
|------|---------|--------------|
| `--output DIR` | `results` | Where the table (`table5_results.csv`/`.json`) and `figures/` are written. |
| `--data PATH` | (auto) | CSV file or folder to use. If omitted, it loads `data/cicids2017_cleaned.csv`; if that is missing it falls back to raw Kaggle (78 features, prints a warning). |
| `--fraction F` | `None` | Keep a stratified fraction in (0,1] of the data (class balance preserved). Smaller = faster, fewer rows. `1.0`/omitted = full dataset. |
| `--nrows N` | `None` | Hard cap of random raw rows read before cleaning (quick dev sanity). |
| `--seed N` | `42` | Global random seed (splits, SMOTE, models). |
| `--optimize` | off | Tune RF/NN/SVM with Optuna instead of fixed hyperparameters. |
| `--n_trials N` | `20` | Optuna trials per baseline (only with `--optimize`). |
| `--parallel_trials N` | `1` | Concurrent Optuna trials for the MLP (CPU-only). `>1` is faster but makes the MLP search non-deterministic. RF/SVM ignore it. |
| `--verbose {0,1,2}` | `1` | Console detail: `0` = results table only, `1` = announce every step (default), `2` = also dump environment/config, per-feature MI scores, library warnings and Optuna trial logs. |
| `--resume` | off | Reuse checkpoints in `--checkpoint-dir` and skip phases already finished (preprocessing + each completed model). Lets a killed run continue. |
| `--checkpoint-dir DIR` | `<output>/checkpoints` | Where phase checkpoints live. Point it **outside the repo** (e.g. `/content/ckpt`) so it survives a re-clone. |

### Checkpoints and resuming

The run saves a checkpoint after preprocessing and after each model, so you do
not have to start from scratch if the Colab session drops or you stop it. To
make checkpoints survive a re-clone of the repo, write them outside the repo
(the cells below use `/content/ckpt`). Add `--resume` to continue.

Note: if you change the data parameters (`--fraction`, `--seed`, `--data`,
`--nrows`) or edit pipeline code, delete the checkpoint dir so stale state is
not reused. The run prints a warning if `--resume` is used with different
parameters than the saved checkpoint.

### Two run modes (cells below)

- Quick pass: a small stratified fraction, no Optuna. A few minutes; produces
  the full table and all figures end to end.
- Final run: a larger fraction with Optuna. Heavier; raise `--fraction` toward
  `1.0` for the whole dataset.

In [ ]:
# Quick pass: small stratified fraction, no Optuna. Finishes in minutes.
# Checkpoints go to /content/ckpt (outside the repo, survives a re-clone).
!python reproduce/run_comparison.py --output results --fraction 0.05 --checkpoint-dir /content/ckpt

In [ ]:
# Final run for the reported results: FULL cleaned dataset + Optuna tuning.
# This is the heaviest run (the reported numbers come from the full dataset, not
# a fraction). It is long; checkpoints let you resume with --resume if it stops.
!python reproduce/run_comparison.py --output results --optimize --n_trials 10 --checkpoint-dir /content/ckpt_final

## 5. Show the results

In [ ]:
import pandas as pd
from IPython.display import Image, display
print(pd.read_csv('results/table5_results.csv').to_string(index=False))
for fig in ['fig_radar_metrics', 'fig_hif_confusion_matrix', 'fig_bar_balanced_acc', 'fig_bar_precision']:
    display(Image('results/figures/%s.png' % fig))